# 23 — Replication Specifies Availability

**Leading specification:** verified chunks become available when they exist as independent replicas.

Notebook 17 split a large artifact into independently verifiable chunks:

\[
I_i=H(c_i)
\]

Notebook 23 asks the next distribution question:

> Once a chunk has an identity, where should it exist?

Replication answers by storing the same verified chunk across independent locations. A replica may be a server copy, mirror, peer cache, archive, or local store. The identity remains the same because the bytes remain the same.

## 1. Context

A verified chunk is useful only if it can be retrieved.

Chunking creates transfer units. Replication creates availability for those transfer units.

```text
verified chunk
→ multiple independent replicas
→ retrieve from any available location
→ verify after retrieval
```

Notebook 23 deliberately avoids peer discovery and routing. Those are later network specifications. Here the focus is the storage-side rule:

> many independent copies preserve access better than one copy.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Robust paths whether run from repo root or notebooks/.
CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = Path("..").resolve()

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "23_replication"
SITE = REPO_ROOT / "site"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
SITE.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from open_model_distribution.hashing import content_id, sha256_file
except Exception:
    def sha256_file(path, chunk_size=1024 * 1024):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def content_id(path, algorithm="sha256"):
        if algorithm != "sha256":
            raise ValueError("Only sha256 is currently supported.")
        return f"sha256:{sha256_file(path)}"

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

print(f"Repo root: {REPO_ROOT}")
print(f"Results:   {RESULTS}")

## 2. Replication variables

Chunk identity remains:

\[
I_i=H(c_i)
\]

Replica locations are represented as:

\[
L_i=\{\ell_{i,1},\ell_{i,2},\ldots,\ell_{i,R_i}\}
\]

where \(R_i=|L_i|\) is the number of replicas for chunk \(i\).

In the simplest specification:

\[
A_i\propto R_i
\]

Availability improves when a chunk has more independent replicas.

In [ ]:
@dataclass(frozen=True)
class ReplicationVariables:
    chunk_identity: str
    replica_locations: str
    replica_count: str
    availability: str
    verification: str

variables = ReplicationVariables(
    chunk_identity="I_i = H(c_i), the digest identity of chunk i",
    replica_locations="L_i = independent storage locations for chunk i",
    replica_count="R_i = |L_i|",
    availability="A_i grows with independent replica count",
    verification="Each replica must hash back to the same chunk identity",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})

## 3. Create one verified chunk

The notebook uses one deterministic chunk as the object being replicated. Later notebooks can apply the same rule across full chunk manifests.

In [ ]:
artifact_dir = RESULTS / "artifacts"
replica_root = RESULTS / "replicas"
retrieval_dir = RESULTS / "retrieval"

for d in [artifact_dir, replica_root, retrieval_dir]:
    d.mkdir(parents=True, exist_ok=True)

chunk_path = artifact_dir / "chunk_0002.bin"
chunk_bytes = (
    b"open-model-distribution replication notebook\n"
    + b"verified chunk payload\n"
    + bytes(range(256)) * 16
)
chunk_path.write_bytes(chunk_bytes)

chunk_digest = sha256_file(chunk_path)
chunk_cid = content_id(chunk_path)

chunk_identity = {
    "chunk_index": 2,
    "path": str(chunk_path.relative_to(REPO_ROOT)),
    "size_bytes": chunk_path.stat().st_size,
    "algorithm": "sha256",
    "digest": chunk_digest,
    "content_id": chunk_cid,
}

chunk_identity_path = RESULTS / "23_chunk_identity.json"
chunk_identity_path.write_text(json.dumps(chunk_identity, indent=2), encoding="utf-8")

chunk_identity

## 4. Replicate the chunk

The same chunk is copied to several independent locations. In this toy example, independence is represented by different replica classes:

- server
- mirror
- peer
- archive
- local cache

In [ ]:
replica_specs = [
    {"replica": "server_A", "class": "server"},
    {"replica": "mirror_B", "class": "mirror"},
    {"replica": "peer_C", "class": "peer"},
    {"replica": "archive_D", "class": "archive"},
    {"replica": "cache_E", "class": "local cache"},
]

replica_rows = []
for spec in replica_specs:
    dst_dir = replica_root / spec["replica"]
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "chunk_0002.bin"
    shutil.copyfile(chunk_path, dst)
    replica_rows.append({
        "chunk_index": 2,
        "replica": spec["replica"],
        "replica_class": spec["class"],
        "path": str(dst.relative_to(REPO_ROOT)),
        "digest": sha256_file(dst),
        "matches_identity": sha256_file(dst) == chunk_digest,
    })

replica_df = pd.DataFrame(replica_rows)
replica_locations_csv = RESULTS / "23_replica_locations.csv"
replica_df.to_csv(replica_locations_csv, index=False)

replica_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.axis("off")
ax.set_title("Replica map", fontsize=17, pad=18)

ax.text(0.15, 0.55, "chunk 2\n" + chunk_digest[:12] + "…", ha="center", va="center", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.5),
        transform=ax.transAxes)

positions = [(0.52, 0.82), (0.72, 0.68), (0.78, 0.45), (0.62, 0.24), (0.42, 0.36)]
for (_, row), (x, y) in zip(replica_df.iterrows(), positions):
    ax.text(x, y, f"{row['replica']}\n{row['replica_class']}", ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    ax.annotate("", xy=(x - 0.10, y), xytext=(0.25, 0.55),
                arrowprops=dict(arrowstyle="->", lw=1.7), xycoords=ax.transAxes)

ax.text(0.5, 0.08, "same identity, multiple independent storage locations", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
replica_map_fig = FIGURES / "23_replica_map.png"
fig.savefig(replica_map_fig, dpi=180)
plt.show()

replica_map_fig

## 5. Verify each replica

Replication does not replace verification. Every replica must hash back to the same chunk identity.

\[
H(c_i^{(\ell_j)})=I_i
\]

In [ ]:
replica_verification_df = replica_df.copy()
replica_verification_df["verification"] = replica_verification_df["matches_identity"].map({True: "PASS", False: "FAIL"})

replica_verification_csv = RESULTS / "23_replica_verification.csv"
replica_verification_df.to_csv(replica_verification_csv, index=False)

replica_verification_df[["replica", "replica_class", "digest", "verification"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.axis("off")
ax.set_title("Replica verification", fontsize=17, pad=16)

for idx, row in replica_verification_df.iterrows():
    x = 0.12 + idx * 0.18
    ax.text(x, 0.60, row["replica"], ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)
    ax.text(x, 0.38, "✓ same digest", ha="center", fontsize=12, transform=ax.transAxes)

ax.text(0.5, 0.14, chunk_digest[:24] + "…", ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
replica_verification_fig = FIGURES / "23_replica_verification.png"
fig.savefig(replica_verification_fig, dpi=180)
plt.show()

replica_verification_fig

## 6. Replica failure

If one location fails, the chunk remains retrievable from other replicas.

Replication changes a single-location failure into a replica-selection problem.

In [ ]:
failed_replica = "mirror_B"

failure_rows = []
for _, row in replica_df.iterrows():
    available = row["replica"] != failed_replica
    failure_rows.append({
        "replica": row["replica"],
        "replica_class": row["replica_class"],
        "available_after_failure": available,
        "digest": row["digest"],
        "matches_identity": row["matches_identity"],
    })

failure_df = pd.DataFrame(failure_rows)
failure_csv = RESULTS / "23_replica_failure.csv"
failure_df.to_csv(failure_csv, index=False)

failure_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
values = failure_df["available_after_failure"].astype(int)
ax.bar(failure_df["replica"], values)
ax.set_ylim(0, 1.2)
ax.set_ylabel("available")
ax.set_yticks([0, 1])
ax.set_yticklabels(["down", "up"])
ax.set_title("One replica can fail without losing the chunk")

for idx, row in failure_df.iterrows():
    label = "failed" if not row["available_after_failure"] else "reachable"
    ax.text(idx, values.iloc[idx] + 0.05, label, ha="center", va="bottom", fontsize=9)

fig.tight_layout()
replica_failure_fig = FIGURES / "23_replica_failure.png"
fig.savefig(replica_failure_fig, dpi=180)
plt.show()

replica_failure_fig

## 7. Select an available replica, then verify

Replica selection is not trust. Selection chooses a source. Verification decides whether the bytes are acceptable.

In [ ]:
available_replicas = failure_df[failure_df["available_after_failure"]].merge(
    replica_df[["replica", "path"]], on="replica", how="left"
)

selected = available_replicas.iloc[0]
selected_path = REPO_ROOT / selected["path"]

retrieved_path = retrieval_dir / "retrieved_chunk_0002.bin"
shutil.copyfile(selected_path, retrieved_path)

retrieval_record = {
    "requested_chunk_index": 2,
    "selected_replica": selected["replica"],
    "selected_replica_class": selected["replica_class"],
    "retrieved_path": str(retrieved_path.relative_to(REPO_ROOT)),
    "retrieved_digest": sha256_file(retrieved_path),
    "expected_digest": chunk_digest,
    "verification": "PASS" if sha256_file(retrieved_path) == chunk_digest else "FAIL",
}

replica_selection_path = RESULTS / "23_replica_selection.json"
replica_selection_path.write_text(json.dumps(retrieval_record, indent=2), encoding="utf-8")

retrieval_record

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.axis("off")
ax.set_title("Replica selection", fontsize=17, pad=16)

nodes = [
    ("request\nchunk 2", 0.12),
    ("available\nreplicas", 0.34),
    ("select\nsource", 0.55),
    ("download\nbytes", 0.74),
    ("verify\ndigest", 0.90),
]

for label, x in nodes:
    ax.text(x, 0.58, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes)

for (_, x1), (_, x2) in zip(nodes[:-1], nodes[1:]):
    ax.annotate("", xy=(x2 - 0.06, 0.58), xytext=(x1 + 0.06, 0.58),
                arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

ax.text(0.5, 0.22, f"selected: {selected['replica']} → verification {retrieval_record['verification']}",
        ha="center", fontsize=12, transform=ax.transAxes)

fig.tight_layout()
replica_selection_fig = FIGURES / "23_replica_selection.png"
fig.savefig(replica_selection_fig, dpi=180)
plt.show()

replica_selection_fig

## 8. Availability from independent replicas

Notebook 23 uses a deliberately simple availability specification:

\[
A_i \propto R_i
\]

This is not a production reliability model. It is a design statement: one copy is fragile; independent copies increase reachability.

In [ ]:
availability_rows = []
for R in range(1, 9):
    availability_rows.append({
        "replicas_R": R,
        "toy_availability_score": R,
        "single_failure_survivable": R > 1,
        "two_failures_survivable": R > 2,
    })

availability_df = pd.DataFrame(availability_rows)
availability_csv = RESULTS / "23_replica_availability.csv"
availability_df.to_csv(availability_csv, index=False)

availability_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(availability_df["replicas_R"], availability_df["toy_availability_score"], marker="o")
ax.set_xlabel("independent replicas R")
ax.set_ylabel("toy availability score")
ax.set_title("Replication increases chunk availability")
ax.set_xticks(availability_df["replicas_R"])
fig.tight_layout()

replication_availability_fig = FIGURES / "23_replication_availability.png"
fig.savefig(replication_availability_fig, dpi=180)
plt.show()

replication_availability_fig

## 9. Replication manifest

The replication manifest records where verified copies of the same chunk exist.

In [ ]:
replica_manifest = {
    "schema": "open-model-distribution.replica-manifest.v0",
    "chunk_index": 2,
    "chunk_identity": chunk_cid,
    "chunk_digest": chunk_digest,
    "replica_count": len(replica_df),
    "replicas": replica_df[["replica", "replica_class", "path", "digest", "matches_identity"]].to_dict(orient="records"),
    "statement": "Same chunk identity, multiple independent storage locations.",
}

replica_manifest_path = RESULTS / "23_replica_manifest.json"
replica_manifest_path.write_text(json.dumps(replica_manifest, indent=2), encoding="utf-8")

replica_manifest

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axis("off")
ax.set_title("Replication manifest", fontsize=17, pad=16)

ax.text(0.20, 0.76, "chunk identity", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.24, 0.76, chunk_cid[:30] + "…", ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.20, 0.64, "replica count", ha="right", va="center", fontsize=12, weight="bold", transform=ax.transAxes)
ax.text(0.24, 0.64, str(len(replica_df)), ha="left", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=1.0),
        transform=ax.transAxes)

ax.text(0.62, 0.76, "replicas", ha="center", va="center", fontsize=13, weight="bold", transform=ax.transAxes)

y = 0.64
for _, row in replica_df.iterrows():
    ax.text(0.46, y, row["replica"], ha="left", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.20", fc="white", ec="black", lw=1.0),
            transform=ax.transAxes)
    ax.text(0.66, y, row["replica_class"], ha="left", va="center", fontsize=11, transform=ax.transAxes)
    y -= 0.10

fig.tight_layout()
replica_manifest_fig = FIGURES / "23_replica_manifest.png"
fig.savefig(replica_manifest_fig, dpi=180)
plt.show()

replica_manifest_fig

## 10. Engineering summary

| Property | Specification | Notebook role |
|---|---|---|
| Identity | Chunk digest | The object stays the same across replicas. |
| Replication | Multiple copies | Availability increases with independent storage locations. |
| Verification | Local digest check | Every replica must hash back to the same identity. |
| Failure | Replica loss | Retrieve from another location. |
| Selection | Available source choice | Selection chooses a source; verification decides trust. |

## 11. Engineering statement

Replication specifies availability by storing the same verified chunk identity across multiple independent locations. Replica selection provides a source for retrieval, but verification remains local: a replica is accepted only when its bytes hash back to the published chunk identity.

## 12. Key equations

Chunk identity:

\[
I_i=H(c_i)
\]

Replica locations:

\[
L_i=\{\ell_{i,1},\ell_{i,2},\ldots,\ell_{i,R_i}\}
\]

Replica count:

\[
R_i=|L_i|
\]

Availability specification:

\[
A_i\propto R_i
\]

Replica verification:

\[
H(c_i^{(\ell_j)})=I_i
\]

## 13. Notebook outputs

Notebook 23 writes:

- `results/23_replication/23_chunk_identity.json`
- `results/23_replication/23_replica_manifest.json`
- `results/23_replication/23_replica_locations.csv`
- `results/23_replication/23_replica_verification.csv`
- `results/23_replication/23_replica_failure.csv`
- `results/23_replication/23_replica_selection.json`
- `results/23_replication/23_replica_availability.csv`
- `figures/23_replica_map.png`
- `figures/23_replica_verification.png`
- `figures/23_replica_failure.png`
- `figures/23_replica_selection.png`
- `figures/23_replication_availability.png`
- `figures/23_replica_manifest.png`

In [ ]:
notebook_manifest = {
    "notebook": "23_replication.ipynb",
    "title": "Replication Specifies Availability",
    "primary_specification": "replication",
    "statement": "Verified chunks become available when they exist as independent replicas.",
    "equations": [
        "I_i=H(c_i)",
        "L_i={ell_i1,...,ell_iR}",
        "R_i=|L_i|",
        "A_i proportional to R_i",
        "H(c_i^(ell_j))=I_i",
    ],
    "outputs": {
        "chunk_identity": str(chunk_identity_path.relative_to(REPO_ROOT)),
        "replica_manifest": str(replica_manifest_path.relative_to(REPO_ROOT)),
        "replica_locations_csv": str(replica_locations_csv.relative_to(REPO_ROOT)),
        "replica_verification_csv": str(replica_verification_csv.relative_to(REPO_ROOT)),
        "replica_failure_csv": str(failure_csv.relative_to(REPO_ROOT)),
        "replica_selection": str(replica_selection_path.relative_to(REPO_ROOT)),
        "replica_availability_csv": str(availability_csv.relative_to(REPO_ROOT)),
        "replica_map_figure": str(replica_map_fig.relative_to(REPO_ROOT)),
        "replica_verification_figure": str(replica_verification_fig.relative_to(REPO_ROOT)),
        "replica_failure_figure": str(replica_failure_fig.relative_to(REPO_ROOT)),
        "replica_selection_figure": str(replica_selection_fig.relative_to(REPO_ROOT)),
        "replication_availability_figure": str(replication_availability_fig.relative_to(REPO_ROOT)),
        "replica_manifest_figure": str(replica_manifest_fig.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "23_notebook_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")

notebook_manifest

## 14. Download artifacts

Run the final cell to package notebook 23 outputs for download.

In [ ]:
# Package notebook artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

NOTEBOOKS = REPO_ROOT / "notebooks"
SITE = REPO_ROOT / "site"

zip_path = RESULTS / "23_replication_artifacts.zip"

notebook_path = NOTEBOOKS / "23_replication.ipynb"
fallback_notebook_path = Path.cwd() / "23_replication.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)

        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)

        elif item.exists() and item.resolve() != zip_path.resolve():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")

display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass